In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
src_path = '/Volumes/scd2/volume/scd2hist/data'
trg_path = '/Volumes/scd2/volume/bronze'
schema_path = "/Volumes/scd2/volume/bronze/_schema"
checkpoint_path = "/Volumes/scd2/volume/bronze/_checkpoint"

In [0]:
df=(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",'csv')
    .option("cloudFiles.schemaLocation",schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Reject files with new columns
    .option("header",True) # Use explicit schema with NOT NULL constraints
    .load(src_path))


In [0]:
import re
def clean_col(col_name:str)->str:
    return re.sub(r'[,;{}()]+','_',col_name).strip('_')

In [0]:
df=df.toDF(*[clean_col(col) for col in df.columns])

In [0]:
query= (df.writeStream.
        format('delta')
        .option("checkpointLocation",checkpoint_path)
        .option('mergeSchema', True)  # Prevent schema evolution - no new columns allowed
        .trigger(once=True)
        .start(trg_path))

# Wait for the stream to complete before continuing
query.awaitTermination()
print("✓ Streaming write completed successfully")

In [0]:
# Read the Delta table from the bronze path
bronze_df = spark.read.format('delta').load(trg_path)

# Display the data
display(bronze_df)

In [0]:
bronze_df.count()